# 07. Applicability Domain

This notebook covers the seventh step in a typical QSAR workflow:
- Evaluating the applicability domain (AD) of the QSAR model
- Determining the chemical space where predictions are reliable
- Identifying compounds outside the applicability domain
- Visualizing AD boundaries

The applicability domain defines the limitations of a QSAR model in terms of structural, physicochemical, and response space. Predictions for compounds within the AD are considered reliable, while those outside should be interpreted with caution.

ProQSAR supports several AD methods:
- k-Nearest Neighbors (k-NN)
- Local Outlier Factor (LOF)
- One-Class SVM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from proqsar.Evaluation.applicability_domain import ApplicabilityDomain
from proqsar.Config.config import Config
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 7.1 Load Data and Model

Load the training and test datasets, and the trained model.

In [ ]:
# Load training and test data
train_path = '../Project/train_data.csv'
test_path = '../Project/test_data.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f"Training data: {df_train.shape}")
print(f"Test data: {df_test.shape}")

In [ ]:
# Prepare data
feature_cols = [col for col in df_train.columns if col not in ['Smiles', 'pChEMBL']]

X_train = df_train[feature_cols]
y_train = df_train['pChEMBL']
X_test = df_test[feature_cols]
y_test = df_test['pChEMBL']

print(f"\nTraining features: {X_train.shape}")
print(f"Test features: {X_test.shape}")

## 7.2 Configure Applicability Domain

Configure the AD method and parameters.

In [ ]:
# Configure applicability domain
config = Config(
    applicability_domain={
        "method": "lof",  # Options: "knn", "lof", "ocsvm"
        "n_neighbors": 5,  # For knn and lof methods
        "rate_of_outliers": 0.1,  # Expected proportion of outliers
    }
)

print("Applicability Domain configured!")
print(f"Method: {config.ad_config['method']}")
print(f"Parameters: {config.ad_config}")

## 7.3 Fit Applicability Domain

Fit the AD model on the training set.

In [ ]:
# Initialize and fit Applicability Domain
ad = ApplicabilityDomain(
    activity_col="pChEMBL",
    id_col="Smiles",
    method=config.ad_config['method'],
    n_neighbors=config.ad_config.get('n_neighbors', 5),
    rate_of_outliers=config.ad_config.get('rate_of_outliers', 0.1),
    save_dir="../Project/ApplicabilityDomain"
)

print("Fitting Applicability Domain on training set...")
ad.fit(df_train)
print("AD model fitted successfully!")

## 7.4 Evaluate Training Set

Check which training compounds are within/outside the AD.

In [ ]:
# Predict AD for training set
train_ad_results = ad.predict(df_train)

# Count compounds in/out of AD
if 'in_ad' in train_ad_results.columns:
    n_in_ad_train = train_ad_results['in_ad'].sum()
    n_out_ad_train = len(train_ad_results) - n_in_ad_train
    
    print(f"Training Set AD Results:")
    print(f"  Compounds in AD: {n_in_ad_train} ({n_in_ad_train/len(train_ad_results)*100:.1f}%)")
    print(f"  Compounds out of AD: {n_out_ad_train} ({n_out_ad_train/len(train_ad_results)*100:.1f}%)")
    
    # Display some outliers
    if n_out_ad_train > 0:
        print("\nExamples of compounds outside AD:")
        print(train_ad_results[~train_ad_results['in_ad']][['Smiles', 'pChEMBL']].head())
else:
    print("AD predictions not available in expected format.")

## 7.5 Evaluate Test Set

Check which test compounds are within/outside the AD.

In [ ]:
# Predict AD for test set
test_ad_results = ad.predict(df_test)

# Count compounds in/out of AD
if 'in_ad' in test_ad_results.columns:
    n_in_ad_test = test_ad_results['in_ad'].sum()
    n_out_ad_test = len(test_ad_results) - n_in_ad_test
    
    print(f"Test Set AD Results:")
    print(f"  Compounds in AD: {n_in_ad_test} ({n_in_ad_test/len(test_ad_results)*100:.1f}%)")
    print(f"  Compounds out of AD: {n_out_ad_test} ({n_out_ad_test/len(test_ad_results)*100:.1f}%)")
    
    # Display some outliers
    if n_out_ad_test > 0:
        print("\nTest compounds outside AD:")
        print(test_ad_results[~test_ad_results['in_ad']][['Smiles', 'pChEMBL']])
else:
    print("AD predictions not available in expected format.")

## 7.6 Visualize Applicability Domain

Visualize the AD using PCA to project the high-dimensional feature space.

In [ ]:
# Combine train and test data for PCA
X_combined = pd.concat([X_train, X_test], axis=0).values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Split back into train and test
X_train_pca = X_pca[:len(X_train)]
X_test_pca = X_pca[len(X_train):]

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")
print(f"PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"PC2: {pca.explained_variance_ratio_[1]:.2%}")

In [ ]:
# Plot AD visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Training set with AD
if 'in_ad' in train_ad_results.columns:
    in_ad_mask = train_ad_results['in_ad'].values
    
    axes[0].scatter(X_train_pca[in_ad_mask, 0], X_train_pca[in_ad_mask, 1], 
                   c='blue', alpha=0.6, s=50, label='In AD')
    axes[0].scatter(X_train_pca[~in_ad_mask, 0], X_train_pca[~in_ad_mask, 1], 
                   c='red', alpha=0.6, s=50, marker='x', label='Out of AD')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
    axes[0].set_title('Training Set: Applicability Domain')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Plot 2: Test set with AD
if 'in_ad' in test_ad_results.columns:
    test_in_ad_mask = test_ad_results['in_ad'].values
    
    # Show training set as reference
    axes[1].scatter(X_train_pca[:, 0], X_train_pca[:, 1], 
                   c='lightgray', alpha=0.3, s=30, label='Training')
    axes[1].scatter(X_test_pca[test_in_ad_mask, 0], X_test_pca[test_in_ad_mask, 1], 
                   c='green', alpha=0.7, s=50, label='Test (In AD)')
    axes[1].scatter(X_test_pca[~test_in_ad_mask, 0], X_test_pca[~test_in_ad_mask, 1], 
                   c='red', alpha=0.7, s=50, marker='x', label='Test (Out of AD)')
    axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
    axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
    axes[1].set_title('Test Set: Applicability Domain')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7.7 AD Analysis by Activity

Analyze how AD relates to activity values and prediction errors.

In [ ]:
# Load test predictions
test_predictions = pd.read_csv('../Project/test_predictions.csv')

# Merge with AD results
if 'in_ad' in test_ad_results.columns:
    test_combined = test_predictions.copy()
    test_combined['in_ad'] = test_ad_results['in_ad'].values
    
    # Calculate statistics by AD
    print("Prediction Performance by AD Status:")
    print("="*60)
    
    in_ad_data = test_combined[test_combined['in_ad']]
    out_ad_data = test_combined[~test_combined['in_ad']]
    
    if len(in_ad_data) > 0:
        print(f"\nCompounds IN AD ({len(in_ad_data)}):")
        print(f"  Mean Absolute Error: {in_ad_data['absolute_error'].mean():.4f}")
        print(f"  Median Absolute Error: {in_ad_data['absolute_error'].median():.4f}")
        print(f"  Max Absolute Error: {in_ad_data['absolute_error'].max():.4f}")
    
    if len(out_ad_data) > 0:
        print(f"\nCompounds OUT of AD ({len(out_ad_data)}):")
        print(f"  Mean Absolute Error: {out_ad_data['absolute_error'].mean():.4f}")
        print(f"  Median Absolute Error: {out_ad_data['absolute_error'].median():.4f}")
        print(f"  Max Absolute Error: {out_ad_data['absolute_error'].max():.4f}")

In [ ]:
# Visualize prediction errors by AD status
if 'in_ad' in test_ad_results.columns and len(test_combined) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Box plot of absolute errors
    ad_labels = ['In AD', 'Out of AD']
    errors_by_ad = [
        test_combined[test_combined['in_ad']]['absolute_error'].values,
        test_combined[~test_combined['in_ad']]['absolute_error'].values
    ]
    
    axes[0].boxplot([e for e in errors_by_ad if len(e) > 0], 
                    labels=[ad_labels[i] for i, e in enumerate(errors_by_ad) if len(e) > 0])
    axes[0].set_ylabel('Absolute Error')
    axes[0].set_title('Prediction Error Distribution by AD Status')
    axes[0].grid(True, alpha=0.3)
    
    # Scatter plot: Activity vs Error, colored by AD
    in_ad_mask = test_combined['in_ad'].values
    axes[1].scatter(test_combined[in_ad_mask]['pChEMBL'], 
                   test_combined[in_ad_mask]['absolute_error'],
                   alpha=0.6, s=50, label='In AD', color='blue')
    axes[1].scatter(test_combined[~in_ad_mask]['pChEMBL'], 
                   test_combined[~in_ad_mask]['absolute_error'],
                   alpha=0.6, s=50, label='Out of AD', color='red', marker='x')
    axes[1].set_xlabel('Actual pChEMBL')
    axes[1].set_ylabel('Absolute Error')
    axes[1].set_title('Prediction Error vs Activity')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7.8 Save AD Results

Save the AD analysis results.

In [ ]:
# Save AD results for test set
if 'in_ad' in test_ad_results.columns:
    test_ad_output = test_combined[['Smiles', 'pChEMBL', 'pChEMBL_predicted', 
                                     'absolute_error', 'in_ad']].copy()
    test_ad_output.to_csv('../Project/test_ad_results.csv', index=False)
    print(f"AD results saved to: ../Project/test_ad_results.csv")
    
    # Save compounds outside AD
    out_ad_compounds = test_ad_output[~test_ad_output['in_ad']]
    if len(out_ad_compounds) > 0:
        out_ad_compounds.to_csv('../Project/test_outside_ad.csv', index=False)
        print(f"Compounds outside AD saved to: ../Project/test_outside_ad.csv")

## 7.9 Summary

Summarize the applicability domain analysis.

In [ ]:
print("="*70)
print("APPLICABILITY DOMAIN SUMMARY")
print("="*70)
print(f"AD Method: {config.ad_config['method'].upper()}")
print(f"\nTraining Set ({len(df_train)} compounds):")
if 'in_ad' in train_ad_results.columns:
    print(f"  In AD: {n_in_ad_train} ({n_in_ad_train/len(train_ad_results)*100:.1f}%)")
    print(f"  Out of AD: {n_out_ad_train} ({n_out_ad_train/len(train_ad_results)*100:.1f}%)")

print(f"\nTest Set ({len(df_test)} compounds):")
if 'in_ad' in test_ad_results.columns:
    print(f"  In AD: {n_in_ad_test} ({n_in_ad_test/len(test_ad_results)*100:.1f}%)")
    print(f"  Out of AD: {n_out_ad_test} ({n_out_ad_test/len(test_ad_results)*100:.1f}%)")

if 'in_ad' in test_ad_results.columns and len(test_combined) > 0:
    in_ad_data = test_combined[test_combined['in_ad']]
    out_ad_data = test_combined[~test_combined['in_ad']]
    
    print(f"\nPrediction Performance:")
    if len(in_ad_data) > 0:
        print(f"  MAE (In AD): {in_ad_data['absolute_error'].mean():.4f}")
    if len(out_ad_data) > 0:
        print(f"  MAE (Out of AD): {out_ad_data['absolute_error'].mean():.4f}")

print(f"\nInterpretation:")
if 'in_ad' in test_ad_results.columns:
    if n_in_ad_test / len(test_ad_results) > 0.8:
        print("  ✓ Most test compounds are within the AD")
    else:
        print("  ⚠ Significant portion of test compounds are outside the AD")
    
    if len(in_ad_data) > 0 and len(out_ad_data) > 0:
        if out_ad_data['absolute_error'].mean() > in_ad_data['absolute_error'].mean():
            print("  ✓ AD is effective: predictions outside AD have higher errors")
        else:
            print("  ⚠ AD may need refinement: errors similar inside/outside AD")

print(f"\nResults saved to:")
print(f"  - ../Project/test_ad_results.csv")
if 'in_ad' in test_ad_results.columns and n_out_ad_test > 0:
    print(f"  - ../Project/test_outside_ad.csv")

print("\nApplicability domain analysis complete!")
print("="*70)

## Conclusion

This completes the typical QSAR workflow:

1. ✓ Data compilation and quality assessment
2. ✓ Molecular descriptor calculation
3. ✓ Feature selection
4. ✓ Dataset splitting
5. ✓ Model building
6. ✓ Model validation
7. ✓ Applicability domain evaluation

The QSAR model is now ready for use in predicting activities of new compounds, with clear understanding of its performance characteristics and the range of chemical structures for which it can make reliable predictions.